In [ ]:
# Install Required Libraries for RAG, Local LLM and Gradio UI
!pip install -q pymupdf faiss-cpu sentence-transformers transformers accelerate gradio joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.6 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 1. Load data, Train RAG Model, and Save Model
# Project: MFU Computer Engineering RAG Assistant
# ==========================================

import os
import re
import fitz
import faiss
import torch
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. Load Dataset PDF
# ==========================================

pdf_file = "/content/sample_data/Computer Engineering _ School of Applied Digital Technology - Mae Fah Luang University.pdf"

if not os.path.exists(pdf_file):
    raise FileNotFoundError(
        "PDF file not found. Please put the PDF file in this path:\n"
        "/content/sample_data/Computer Engineering _ School of Applied Digital Technology - Mae Fah Luang University.pdf"
    )

print("Loading PDF dataset...")

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

document = fitz.open(pdf_file)

text_data = ""

for page_number, page in enumerate(document, start=1):
    page_text = page.get_text("text")
    page_text = clean_text(page_text)

    if page_text:
        text_data += f"""
Page {page_number}
{page_text}
--------------------------------------------------
"""

document.close()

print("PDF text extracted successfully.")
print("Total characters:", len(text_data))
print("Preview of dataset:")
print(text_data[:1000])

# ==========================================
# 2. Split Text into Chunks
# ==========================================

print("Splitting text into chunks...")

def split_text(text, chunk_size=500, chunk_overlap=100):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - chunk_overlap

    return chunks

chunks = split_text(text_data)

print("Number of chunks:", len(chunks))

# ==========================================
# 3. Create Embeddings
# ==========================================

print("Creating embeddings...")

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    embedding_model_name,
    device=device
)

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

# ==========================================
# 4. Train FAISS Vector Model
# ==========================================

print("Training FAISS vector model...")

dimension = embeddings.shape[1]

vectorstore = faiss.IndexFlatIP(dimension)
vectorstore.add(embeddings)

print("FAISS vector model trained successfully.")
print("Total vectors:", vectorstore.ntotal)

# ==========================================
# 5. Save RAG Model and Scaler
# ==========================================

project_name = "MFU Computer Engineering RAG Assistant"
group_no = "BDA_Project2_14"

group_members = [
    "6631501135 — Min Maung Maung",
    "6631501134 — Kyaw Thura",
    "6631501186 — Kyaw Phone Thu",
    "6631501183 — Phone Wai Yan Thaing"
]

rag_model = {
    "vectorstore": vectorstore,
    "chunks": chunks,
    "project_name": project_name,
    "group_no": group_no,
    "group_members": group_members
}

scaler = {
    "embedding_model_name": embedding_model_name
}

joblib.dump(rag_model, "best_ce_rag_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model saved successfully: best_ce_rag_model.pkl")
print("Scaler saved successfully: scaler.pkl")

Loading PDF dataset...
PDF text extracted successfully.
Total characters: 18280
Preview of dataset:

Page 1
Computer Engineering TQF.2 | TQF.3 | TABEE 2567 Bachelor of Engineering Program in Computer Engineering Degree Title: Full Title: Bachelor of Engineering (Computer Engineering) Abbreviation: B.Eng. (Computer Engineering) 4/18/26, 9:48 AM Computer Engineering | School of Applied Digital Technology - Mae Fah Luang University https://adt.mfu.ac.th/en/it-computer-engineering.html 1/20
--------------------------------------------------

Page 2
Program Overview: The Bachelor of Engineering Program in Computer Engineering aims to produce graduates equipped with world-class theoretical knowledge and practical expertise. Our students develop specialized skills in designing, planning, and applying emerging technologies to create innovative solutions. The curriculum covers the comprehensive maintenance of both computer hardware and software systems. To meet the demands of modern industries 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Training FAISS vector model...
FAISS vector model trained successfully.
Total vectors: 6
Model saved successfully: best_ce_rag_model.pkl
Scaler saved successfully: scaler.pkl


In [ ]:
# ==========================================
# 2. Test the Trained RAG Model
# ==========================================

import joblib
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load saved model and scaler
rag_model = joblib.load("best_ce_rag_model.pkl")
scaler = joblib.load("scaler.pkl")

vectorstore = rag_model["vectorstore"]
chunks = rag_model["chunks"]

embedding_model_name = scaler["embedding_model_name"]

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load embedding model
embedding_model = SentenceTransformer(
    embedding_model_name,
    device=device
)

# Setup the Local LLM
print("Loading local HuggingFace model...")

model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
llm_model = llm_model.to(device)

# Test question
question = "What are the three learning tracks in Computer Engineering?"

# Retrieve relevant chunks
question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = vectorstore.search(question_embedding, 3)

retrieved_chunks = []

for score, idx in zip(scores[0], indices[0]):
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=200
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", question)
print("=" * 80)
print("Answer:", answer)
print("=" * 80)

for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\nRetrieved Chunk {i}")
    print("Similarity Score:", round(float(scores[0][i-1]), 4))
    print(chunk[:1000])
    print("-" * 80)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading local HuggingFace model...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: What are the three learning tracks in Computer Engineering?
Answer: Computer Engineering

Retrieved Chunk 1
Similarity Score: 0.6146
Page 1 Computer Engineering TQF.2 | TQF.3 | TABEE 2567 Bachelor of Engineering Program in Computer Engineering Degree Title: Full Title: Bachelor of Engineering (Computer Engineering) Abbreviation: B.Eng. (Computer Engineering) 4/18/26, 9:48 AM Computer Engineering | School of Applied Digital Technology - Mae Fah Luang University https://adt.mfu.ac.th/en/it-computer-engineering.html 1/20 -------------------------------------------------- Page 2 Program Overview: The Bachelor of Engineering Program in Computer Engineering aims to produce graduates equipped with world-class theoretical knowledge and practical expertise. Our students develop specialized skills in designing, planning, and applying emerging technologies to create innovative solutions. The curriculum covers the comprehensive maintenance of both computer hardware and software systems. 

In [ ]:
# ==========================================
# 3. Build and Launch the Gradio UI
# ==========================================

import gradio as gr
import joblib
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("\n--- Launching MFU Computer Engineering RAG Assistant UI ---")

# Load saved RAG model
rag_model = joblib.load("best_ce_rag_model.pkl")
scaler = joblib.load("scaler.pkl")

vectorstore = rag_model["vectorstore"]
chunks = rag_model["chunks"]

embedding_model_name = scaler["embedding_model_name"]

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load embedding model
embedding_model = SentenceTransformer(
    embedding_model_name,
    device=device
)

# Load local LLM
model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
llm_model = llm_model.to(device)

def chat_with_ce_guide(message, history):
    # Convert user question into embedding
    question_embedding = embedding_model.encode(
        [message],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve relevant chunks from FAISS model
    scores, indices = vectorstore.search(question_embedding, 3)

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):
        retrieved_chunks.append(chunks[idx])

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are an academic assistant for the Computer Engineering Program at Mae Fah Luang University.

Answer the question using only the context below.
If the answer is not found in the context, say:
The information is not available in the provided document.

Context:
{context}

Question:
{message}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

ui = gr.ChatInterface(
    fn=chat_with_ce_guide,
    title="MFU COMPUTER ENGINEERING RAG ASSISTANT",
    description="""
    Ask me questions based on the Computer Engineering Program PDF dataset.

    Group No.: BDA_Project2_14

    Group Members:
    6631501135 — Min Maung Maung
    6631501134 — Kyaw Thura
    6631501186 — Kyaw Phone Thu
    6631501183 — Phone Wai Yan Thaing
    """,
    examples=[
        "What is the Computer Engineering program about?",
        "What is the degree title?",
        "What are the three learning tracks in Computer Engineering?",
        "What career opportunities are available after graduation?",
        "How much is the total program fee?",
        "What is the curriculum structure?",
        "What are the Program Learning Outcomes?",
        "What are the Program Educational Objectives?",
        "Why should students choose Computer Engineering at MFU?",
        "What is the contact information?"
    ]
)

ui.launch(share=True, debug=True)


--- Launching MFU Computer Engineering RAG Assistant UI ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0cdbae851b6e9ebb77.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# PLEASE DEPLOY this source code for Gradio

%%writefile app.py
import gradio as gr
import joblib
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==========================================
# 1. Load the trained RAG model and scaler
# ==========================================

rag_model = joblib.load("best_ce_rag_model.pkl")
scaler = joblib.load("scaler.pkl")

vectorstore = rag_model["vectorstore"]
chunks = rag_model["chunks"]

embedding_model_name = scaler["embedding_model_name"]

# ==========================================
# 2. Setup the Embedding Model and Local LLM
# ==========================================

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    embedding_model_name,
    device=device
)

model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
llm_model = llm_model.to(device)

# ==========================================
# 3. Core Chatbot Function
# ==========================================

def chat_with_ce_guide(message, history):
    question_embedding = embedding_model.encode(
        [message],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = vectorstore.search(question_embedding, 3)

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):
        retrieved_chunks.append(chunks[idx])

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are an academic assistant for the Computer Engineering Program at Mae Fah Luang University.

Answer the question using only the context below.
If the answer is not found in the context, say:
The information is not available in the provided document.

Context:
{context}

Question:
{message}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

# ==========================================
# 4. Build the Interactive Gradio Interface
# ==========================================

ui = gr.ChatInterface(
    fn=chat_with_ce_guide,
    title="MFU COMPUTER ENGINEERING RAG ASSISTANT",
    description="""
    This chatbot answers questions about the Computer Engineering Program,
    School of Applied Digital Technology, Mae Fah Luang University.

    6631501135 — Min Maung Maung
    """,
    examples=[
        "What is the Computer Engineering program about?",
        "What is the degree title?",
        "What are the three learning tracks in Computer Engineering?",
        "What career opportunities are available after graduation?",
        "How much is the total program fee?",
        "What is the curriculum structure?",
        "What are the Program Learning Outcomes?",
        "What are the Program Educational Objectives?",
        "Why should students choose Computer Engineering at MFU?",
        "What is the contact information?"
    ]
)

# ==========================================
# 5. Launch the web app
# ==========================================

ui.launch()

In [ ]:
# Create the requirements.txt file

%%writefile requirements.txt
gradio
joblib
torch
transformers
accelerate
faiss-cpu
sentence-transformers

# Check all required files for deployment

import os

required_files = [
    "app.py",
    "best_ce_rag_model.pkl",
    "scaler.pkl",
    "requirements.txt"
]

for file in required_files:
    if os.path.exists(file):
        print(file, "✅ Found")
    else:
        print(file, "❌ Missing")